In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

sys.path.append(os.path.dirname(os.getcwd()))
from lib.utils import get_sequence_data
from lib.BLogistic import SkewedBLogistic, train_blogistic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

folder_path = r"../../MarketData/historical_data"
context_window = 60
X, Y = get_sequence_data(folder_path, context_window, force_recompute=True)
dof = 16
simple_X = torch.tensor(X[:, :, 0], device=device)
simple_Y = torch.tensor(Y, device=device).reshape(-1, 1)

dev_size = 10000
np.random.seed(0)
indices = np.random.permutation(simple_X.shape[0])

dev_indices = indices[:dev_size]
train_indices = indices[dev_size:]
train_X = simple_X[train_indices]
train_Y = simple_Y[train_indices]
dev_X = simple_X[dev_indices, :]
dev_Y = simple_Y[dev_indices]

std = train_Y.std()
train_X = train_X / std
train_Y = train_Y / std
dev_X = dev_X / std
dev_Y = dev_Y / std
print("train_X", train_X.shape, "train_Y", train_Y.shape, "dev_X", dev_X.shape, "dev_Y", dev_Y.shape)
print("std", std, train_X.std())

In [ ]:

class LSTMProbNN(nn.Module):
    def __init__(self, context_window, dof, device, hidden_sizes=[128, 64, 32],
                 dropout=0.02, l2_reg=0.002):
        super().__init__()
        self.context_window = context_window
        self.dof = dof
        self.device = device


        # LSTM layers
        self.lstm1 = nn.LSTM(
            input_size=1, hidden_size=hidden_sizes[0],
            batch_first=True, dropout=dropout
        )
        self.lstm2 = nn.LSTM(
            input_size=hidden_sizes[0], hidden_size=hidden_sizes[1],
            batch_first=True, dropout=dropout
        )
        self.lstm3 = nn.LSTM(
            input_size=hidden_sizes[1], hidden_size=hidden_sizes[2],
            batch_first=True, dropout=dropout
        )

        # Fully connected output → distribution parameters
        self.fc = nn.Linear(hidden_sizes[-1], dof)
        nn.init.uniform_(self.fc.weight, -0.01, 0.01)
        nn.init.zeros_(self.fc.bias)

        self.blogistic = SkewedBLogistic(dof - 3, device=device)

        print(f"LSTM initialized: {hidden_sizes}, dropout={dropout}, L2={l2_reg}")

    def forward(self, x):

        x = x.float().to(self.device)
        if x.dim() == 2:
            x = x.unsqueeze(-1)  # (batch, seq, 1)

        out, _ = self.lstm1(x)
        out, _ = self.lstm2(out)
        out, _ = self.lstm3(out)
        out = out[:, -1, :]  # last time step
        params = self.fc(out)
        return params

    def get_params(self, x):
        return self.forward(x)

    def loss_fn(self, x, y):
        params = self.forward(x)
        logpdf = self.blogistic.logpdf_vectorized(
            y, params[:, :-2], params[:, -2], params[:, -1]
        )
        return -logpdf.mean()

In [ ]:
def train_lstm_nn(
    train_X, train_Y, dev_X, dev_Y,
    lr=0.002, num_steps=300,
    device="cuda",
    batch_size=128,
    patience=10
):
    # Convert all to float32 tensors
    train_X, train_Y = train_X.float().to(device), train_Y.float().to(device)
    dev_X, dev_Y = dev_X.float().to(device), dev_Y.float().to(device)

    print(f"Train X: {train_X.shape}, Train Y: {train_Y.shape}")

    # Initialize model
    model = LSTMProbNN(context_window, dof, device).to(device)

    # Optional transfer-learn init (as in your CNN)
    iid_lr = 0.05
    iid_steps = 500
    _, offset = train_blogistic(train_Y.flatten(), dof, iid_lr, iid_steps,
                                allow_skew=True, device=device)
    with torch.no_grad():
        offset = [o.float() for o in offset]
        model.fc.bias[:-2].copy_(offset[0])
        model.fc.bias[-2].copy_(offset[1])
        model.fc.bias[-1].copy_(offset[2])

    # Optimizer with weight decay (L2)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.002)

    best_dev_loss = float("inf")
    train_losses, dev_losses = [], []
    epochs_no_improve = 0

    # Training loop
    for step in range(num_steps):
        model.train()
        idx = np.random.permutation(train_X.shape[0])
        total_loss = 0.0

        for i in range(0, train_X.shape[0], batch_size):
            batch_idx = idx[i:i+batch_size]
            batch_X, batch_Y = train_X[batch_idx], train_Y[batch_idx]

            loss = model.loss_fn(batch_X, batch_Y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / (train_X.shape[0] / batch_size)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        with torch.no_grad():
            dev_loss_sum = 0.0
            for j in range(0, dev_X.shape[0], batch_size):
                batch_X = dev_X[j:j+batch_size]
                batch_Y = dev_Y[j:j+batch_size]
                dev_loss_sum += model.loss_fn(batch_X, batch_Y).item()
            avg_dev_loss = dev_loss_sum / (dev_X.shape[0] / batch_size)
            dev_losses.append(avg_dev_loss)

        # Logging
        if step % 10 == 0 or step == num_steps - 1:
            print(f"[Epoch {step:03d}] Train Loss: {avg_train_loss:.4f} | Dev Loss: {avg_dev_loss:.4f}")

        # Early stopping
        if avg_dev_loss < best_dev_loss:
            best_dev_loss = avg_dev_loss
            torch.save(model.state_dict(), f"best_lstm_model_context{context_window}.pth")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {step}")
                break

    print(f"Best Dev Loss: {best_dev_loss:.4f}")
    return model, train_losses, dev_losses


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# SET DEFAULT DTYPE TO FLOAT32 GLOBALLY
torch.set_default_dtype(torch.float32)

class LSTMProbNN(nn.Module):
    def __init__(self, context_window, dof, device, hidden_sizes=[128, 64, 32],
                 dropout=0.02, l2_reg=0.002):
        super().__init__()
        self.context_window = context_window
        self.dof = dof
        self.device = device

        # LSTM layers
        self.lstm1 = nn.LSTM(
            input_size=1, hidden_size=hidden_sizes[0],
            batch_first=True, dropout=dropout
        )
        self.lstm2 = nn.LSTM(
            input_size=hidden_sizes[0], hidden_size=hidden_sizes[1],
            batch_first=True, dropout=dropout
        )
        self.lstm3 = nn.LSTM(
            input_size=hidden_sizes[1], hidden_size=hidden_sizes[2],
            batch_first=True, dropout=dropout
        )

        # CRITICAL: Convert all LSTM layers to float32
        self.lstm1 = self.lstm1.float()
        self.lstm2 = self.lstm2.float()
        self.lstm3 = self.lstm3.float()

        # Fully connected output → distribution parameters
        self.fc = nn.Linear(hidden_sizes[-1], dof)
        self.fc = self.fc.float()

        nn.init.uniform_(self.fc.weight, -0.01, 0.01)
        nn.init.zeros_(self.fc.bias)

        # BLogistic distribution handler
        self.blogistic = SkewedBLogistic(dof - 3, device=device)

        # Convert all blogistic tensors to float32
        self._convert_blogistic_to_float32()

        print(f"LSTM initialized: {hidden_sizes}, dropout={dropout}, L2={l2_reg}")

    def _convert_blogistic_to_float32(self):
        """Convert all tensors in blogistic to float32"""
        for key in list(vars(self.blogistic).keys()):
            val = getattr(self.blogistic, key)
            if isinstance(val, torch.Tensor) and val.dtype == torch.float64:
                setattr(self.blogistic, key, val.float())

    def forward(self, x):
        x = x.float().to(self.device)
        if x.dim() == 2:
            x = x.unsqueeze(-1)  # (batch, seq, 1)

        out, _ = self.lstm1(x)
        out, _ = self.lstm2(out)
        out, _ = self.lstm3(out)
        out = out[:, -1, :]  # last time step
        params = self.fc(out)
        return params

    def get_params(self, x):
        return self.forward(x)

    def loss_fn(self, x, y):
        params = self.forward(x)
        # Ensure all parameters are float32
        logpdf = self.blogistic.logpdf_vectorized(
            y.float(),
            params[:, :-2].float(),
            params[:, -2].float(),
            params[:, -1].float()
        )
        return -logpdf.mean()

def train_lstm_nn(
    train_X, train_Y, dev_X, dev_Y,
    lr=0.002, num_steps=300,
    device="cuda",
    batch_size=128,
    patience=10
):
    # Convert all to float32 tensors
    train_X = train_X.float().to(device)
    train_Y = train_Y.float().to(device)
    dev_X = dev_X.float().to(device)
    dev_Y = dev_Y.float().to(device)

    print(f"Train X: {train_X.shape}, dtype: {train_X.dtype}")
    print(f"Train Y: {train_Y.shape}, dtype: {train_Y.dtype}")

    # Initialize model
    model = LSTMProbNN(context_window, dof, device).to(device)

    # Optional transfer-learn init
    iid_lr = 0.05
    iid_steps = 500
    _, offset = train_blogistic(train_Y.flatten(), dof, iid_lr, iid_steps,
                                allow_skew=True, device=device)
    with torch.no_grad():
        offset = [o.float() for o in offset]
        model.fc.bias[:-2].copy_(offset[0])
        model.fc.bias[-2].copy_(offset[1])
        model.fc.bias[-1].copy_(offset[2])

    # Optimizer with weight decay (L2)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.002)

    best_dev_loss = float("inf")
    train_losses, dev_losses = [], []
    epochs_no_improve = 0

    # Training loop
    for step in range(num_steps):
        model.train()
        idx = np.random.permutation(train_X.shape[0])
        total_loss = 0.0
        num_batches = 0

        for i in range(0, train_X.shape[0], batch_size):
            batch_idx = idx[i:i+batch_size]
            batch_X = train_X[batch_idx]
            batch_Y = train_Y[batch_idx]

            loss = model.loss_fn(batch_X, batch_Y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_train_loss = total_loss / num_batches
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        with torch.no_grad():
            dev_loss_sum = 0.0
            num_dev_batches = 0
            for j in range(0, dev_X.shape[0], batch_size):
                batch_X = dev_X[j:j+batch_size]
                batch_Y = dev_Y[j:j+batch_size]
                dev_loss_sum += model.loss_fn(batch_X, batch_Y).item()
                num_dev_batches += 1
            avg_dev_loss = dev_loss_sum / num_dev_batches
            dev_losses.append(avg_dev_loss)

        # Logging
        if step % 10 == 0 or step == num_steps - 1:
            print(f"[Epoch {step:03d}] Train Loss: {avg_train_loss:.4f} | Dev Loss: {avg_dev_loss:.4f}")

        # Early stopping
        if avg_dev_loss < best_dev_loss:
            best_dev_loss = avg_dev_loss
            torch.save(model.state_dict(), f"best_lstm_model_context{context_window}.pth")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {step}")
                break

    print(f"Best Dev Loss: {best_dev_loss:.4f}")
    return model, train_losses, dev_losses

In [ ]:
print(f"Context window: {context_window}")
print(f"DOF: {dof}")
print(f"Device: {device}")

# Training
lr = 2e-4
num_steps = 3000
model, train_losses, dev_losses = train_lstm_nn(train_X, train_Y, dev_X, dev_Y, lr, num_steps, device=device)
torch.save(model.state_dict(), f"jl_lstm_model_context{context_window}.pth")

# Loading and plotting
model = LSTMProbNN(context_window, dof, device).to(device)
model.load_state_dict(torch.load(f"jl_ltsm_model_context{context_window}.pth", map_location=device))
model.eval()

plot_xs = torch.linspace(-8, 8, 10000, dtype=torch.float32, device='cpu')
nplots = min(10, dev_X.shape[0])

plt.figure(figsize=(10, 6))
with torch.no_grad():
    for idx in range(nplots):
        color = plt.cm.viridis(idx / nplots)

        # Ensure float32 and move to CPU for plotting
        dev_X_cpu = dev_X[idx, :].reshape(1, -1).float().cpu()
        dev_Y_cpu = dev_Y[idx, :].float().cpu()

        # Use batched computation
        plot_ys = model.get_pdf_batched(dev_X_cpu, plot_xs, batch_size=500)
        plt.plot(plot_xs.numpy(), plot_ys.numpy(), color=color, alpha=0.7)

        point_pdf = model.get_pdf(dev_X_cpu, dev_Y_cpu.reshape(-1))
        plt.scatter(dev_Y_cpu.numpy(), point_pdf.cpu().numpy(), color=color, s=50, zorder=5)

plt.xlabel("Return")
plt.ylabel("PDF")
plt.title("Predicted PDFs")
plt.grid(True, alpha=0.3)
plt.show()